# Milestone 3 - the tutor's branching policy

**Why this matters.** This one function decides everything about how the tutor
feels: how long it lets you struggle, and when it gives up and explains. It is
also the clearest example in the project of a decision worth isolating - no model
call, no I/O, so it is trivially testable and you can reason about it completely.

**New to coding?** Do `m0_python_basics.ipynb` first. Words you'll meet here: pure function, edge case. Each one is explained in plain language in [docs/CONCEPTS.md](../docs/CONCEPTS.md), along with Big-O and a routine for getting started when the cell is blank.

**How to work through this notebook.** Run the cells top to bottom. Each
exercise has a cell for you to fill in, then a cell of tests. Run the tests
first and watch them fail - read *what* they check, because the tests are the
specification. Then write code until they go green.

After you edit an exercise cell, **re-run it** before re-running its tests - the
tests use whatever is currently defined in the notebook.

In [ ]:
import ipytest
import pytest

# clean="test_*": before each test cell, forget test functions from earlier cells
# - but not other names starting with "Test", like milestone 2's TestCase class.
ipytest.autoconfig(clean="test_*")

## `route_after_tests`

Decide what happens after a submission has been graded. Return exactly one of:

| Return | When |
|---|---|
| `"review"` | the tests passed - critique the working solution |
| `"hint"` | the tests failed and attempts remain - nudge them |
| `"explain"` | the tests failed and attempts are used up - walk them through it |

Read the state from these keys:

- `state["result"]["passed"]` - bool (may be missing entirely)
- `state["attempts"]` - int (may be missing)
- `state["max_attempts"]` - int (may be missing; default to 3)

Three details the tests check, each of which is a real bug if you miss it:

1. Passing on the **final** attempt still routes to `"review"`. Solving it on the
   last try is solving it.
2. A missing `result` means the grader never ran, which is a failure - it must
   not be read as a pass.
3. `max_attempts` comes from the state, not a hard-coded 3, so it stays
   configurable.

In [ ]:
def route_after_tests(state: dict) -> str:
    """Decide what happens after a submission has been graded."""
    # TODO: implement.
    raise NotImplementedError("Milestone 3: implement route_after_tests")

In [ ]:
%%ipytest -qq

def state(passed: bool, attempts: int, max_attempts: int = 3) -> dict:
    return {"result": {"passed": passed}, "attempts": attempts, "max_attempts": max_attempts}


def test_passing_routes_to_review():
    assert route_after_tests(state(True, 1)) == "review"


def test_passing_on_the_last_attempt_still_reviews():
    assert route_after_tests(state(True, 3)) == "review"


@pytest.mark.parametrize("attempts", [1, 2])
def test_failing_with_attempts_left_routes_to_hint(attempts):
    assert route_after_tests(state(False, attempts)) == "hint"


@pytest.mark.parametrize("attempts", [3, 4])
def test_failing_out_of_attempts_routes_to_explain(attempts):
    assert route_after_tests(state(False, attempts)) == "explain"


def test_a_missing_result_is_a_failure_not_a_pass():
    assert route_after_tests({"attempts": 1, "max_attempts": 3}) == "hint"


def test_max_attempts_is_read_from_the_state():
    assert route_after_tests(state(False, 3, max_attempts=5)) == "hint"
    assert route_after_tests(state(False, 5, max_attempts=5)) == "explain"


def test_max_attempts_defaults_to_three():
    assert route_after_tests({"result": {"passed": False}, "attempts": 3}) == "explain"


def test_only_the_three_known_values_are_returned():
    assert route_after_tests(state(True, 1)) in {"review", "hint", "explain"}

## Try it

A whole session's routing decisions in one table. Change `max_attempts` and
re-run to see the policy shift.

In [ ]:
max_attempts = 3
for attempts in range(1, max_attempts + 2):
    for passed in (False, True):
        s = {"result": {"passed": passed}, "attempts": attempts, "max_attempts": max_attempts}
        print(f"attempt {attempts}  passed={passed!s:<5}  ->  {route_after_tests(s)}")